# PlantSeg Multi-Seed Retraining -- Classifiers-Only Lane A (seed 123)

One of **2 parallel Kaggle notebooks** covering the classifier half of the 2-new-seed
(3 total, including the original SEED=42 checkpoints) retraining pass for the Array revision's
variance/robustness response to R4/R5. This is a **replacement** for the classifier sections of
the original 6-lane split, which failed on every lane due to a `DATA_ROOT` bug in the classifier
config cell (see below) -- **the segmenter half of that original run already completed
successfully and is not repeated here.**

**This notebook (Lane A, seed 123) trains all 8 classifiers at seed 123:**
ConvNeXtV2Tiny, EfficientNetV2S, ResNet50, EfficientNetB0, MobileNetV3Large, ViTBase16, SwinTiny, GatedFusionLight.
Hyperparameters, architectures, losses and schedulers are all unchanged from the original Stage 1
trainer -- only `SEED` and the output directory differ, so these results are directly comparable
to the existing SEED=42 checkpoints.

**Estimated time:** ~7.7h of core training (sum of each classifier's original single-seed
`train_time_sec` from its `result_*.pkl`), plus data-loading/TTA-eval/deployment-benchmark
overhead -- budget **~9-9.5h**, leaving ~2.5-3h of margin under Kaggle's 12h session limit, in
line with the original 6-lane split's margin philosophy. All 8 classifiers fit in one seed's
lane; the only reason this is split into 2 notebooks rather than 1 is that seed 123 and seed
7 together (~15.5h raw) would not fit in a single 12h session.

**What was fixed vs. the original classifier-section bug:** the original notebooks hardcoded
`DATA_ROOT = Path("/kaggle/input/plantseg/plantseg")` in the classifier config cell, which does
not exist on Kaggle for this dataset attachment -- the real mount path (confirmed from the
segmenter section of the same lanes, which someone had already manually corrected and which DID
run to completion) is `/kaggle/input/datasets/alenmaruf/plantseg/plantseg`. This notebook uses
that verified path as primary, with the old paths kept as fallbacks.

**Attach as a Kaggle dataset:** the PlantSeg image+annotation dataset (Metadata.csv + images/ +
annotation_train.json/annotation_val.json/annotation_test.json). No pretrained checkpoints need
attaching -- classifiers train from ImageNet initialization via `torchvision`/`timm`, same as the
original runs.

**After it finishes:** collect this notebook's output folder (`/kaggle/working/s1_results_seed123/`)
and bring it back to `revision_work/multiseed/` alongside the already-collected seed-123
segmenter results, so both halves of seed 123 can be aggregated together.

In [1]:
# ============ CLASSIFIER SECTION: IMPORTS + SEED ============
!pip install -q timm
import os, json, math, time, random, warnings, pickle
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision; from torchvision import transforms
import timm
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score,
                             recall_score, matthews_corrcoef, confusion_matrix, roc_auc_score)
warnings.filterwarnings("ignore")

SEED = 123   # <-- new seed for this run (original paper checkpoints used SEED=42)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("Device:", device, "| SEED:", SEED)
assert device.type == "cuda", "No GPU detected -- set Settings > Accelerator (T4 x2 / P100) before Run All."

Device: cuda | SEED: 123


In [2]:
# ============ CLASSIFIER SECTION: CONFIG ============
# DATA_ROOT fixed: the original classifier-section block hardcoded a path that does not exist on
# Kaggle for this dataset attachment ("/kaggle/input/plantseg/plantseg"); the verified correct
# path (confirmed by the segmenter section of the same lanes completing successfully) is used as
# primary here, with the old paths kept as fallbacks.
DATA_ROOT = Path("/kaggle/input/datasets/alenmaruf/plantseg/plantseg")
if not (DATA_ROOT/"Metadata.csv").exists():
    for cand in [Path("/kaggle/input/datasets/alenmaruf/plantseg/plantseg"), Path("/kaggle/input/plantseg/plantseg"),
                 Path("/kaggle/input/plantseg"), Path("plantseg"), Path(".")]:
        if (cand/"Metadata.csv").exists(): DATA_ROOT=cand; break
assert (DATA_ROOT/"Metadata.csv").exists(), f"Metadata.csv not found under {DATA_ROOT} -- edit DATA_ROOT."

OUTDIR = Path(f"/kaggle/working/s1_results_seed{SEED}"); FIG = OUTDIR/"figures"
OUTDIR.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

MODELS_TO_TRAIN = ['ConvNeXtV2Tiny', 'EfficientNetV2S', 'ResNet50', 'EfficientNetB0', 'MobileNetV3Large', 'ViTBase16', 'SwinTiny', 'GatedFusionLight']   # all 8 classifiers -- this lane's full assignment

SCOPE = "full"
FRUIT_HOSTS = {"Apple","Citrus","Grape","Banana","Peach","Blueberry",
               "Plum","Cherry","Raspberry","Strawberry"}

NUM_EPOCHS      = 40
PATIENCE        = 8
LR              = 3e-4
WEIGHT_DECAY    = 1e-4
USE_PRETRAINED  = True
USE_CLASS_WEIGHTS = True
LABEL_SMOOTHING = 0.05
IMG_SIZE_CNN    = 384   # lower to 320/256 if OOM -- CNN classifiers trained at 384x384
USE_RANDAUG     = True
USE_TTA         = True
QUICK_SMOKE     = False

print("DATA_ROOT:", DATA_ROOT); print(f"Models to train (seed {SEED}):", MODELS_TO_TRAIN)

DATA_ROOT: /kaggle/input/datasets/alenmaruf/plantseg/plantseg
Models to train (seed 123): ['ConvNeXtV2Tiny', 'EfficientNetV2S', 'ResNet50', 'EfficientNetB0', 'MobileNetV3Large', 'ViTBase16', 'SwinTiny', 'GatedFusionLight']


In [3]:
# ============ CLASSIFIER SECTION: DATA INDEX ============
meta = pd.read_csv(DATA_ROOT/"Metadata.csv", encoding="utf-8-sig")
meta["split"] = meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"] = meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"], axis=1)
meta["label"] = meta["Disease"]; meta["plant"] = meta["Plant"]
if SCOPE == "fruit":
    meta = meta[meta["plant"].isin(FRUIT_HOSTS)].reset_index(drop=True)
meta = meta[meta["filepath"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

CLASS_NAMES = sorted(meta["label"].unique().tolist())
NUM_CLASSES = len(CLASS_NAMES)
label2idx = {c: i for i, c in enumerate(CLASS_NAMES)}

train_df = meta[meta.split=="train"].reset_index(drop=True)
val_df   = meta[meta.split=="val"].reset_index(drop=True)
test_df  = meta[meta.split=="test"].reset_index(drop=True)

counts = train_df["label"].map(label2idx).value_counts().reindex(range(NUM_CLASSES)).fillna(0)
inv = np.where(counts.values > 0, 1.0/counts.values, 0.0)
cw  = inv / inv[inv>0].mean()
class_weights = torch.tensor(cw, dtype=torch.float32, device=device)

print(f"classes={NUM_CLASSES}  train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

classes=115  train=5367  val=846  test=1561


In [4]:
# ============ CLASSIFIER SECTION: TRANSFORMS / DATASET ============
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]

def make_transforms(sz):
    ops = [transforms.RandomResizedCrop(sz, scale=(0.7,1.0)),
           transforms.RandomHorizontalFlip(),
           transforms.RandomVerticalFlip(p=0.2),
           transforms.RandAugment(num_ops=2, magnitude=9),
           transforms.ToTensor(), transforms.Normalize(MEAN, STD),
           transforms.RandomErasing(p=0.25)]
    ev = transforms.Compose([transforms.Resize((sz,sz)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    return transforms.Compose(ops), ev

class DS(Dataset):
    def __init__(self, d, t):
        self.df = d.reset_index(drop=True); self.t = t
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.t(Image.open(r.filepath).convert("RGB")), label2idx[r.label]

In [5]:
# ============ CLASSIFIER SECTION: MODEL ZOO ============
# Verbatim from Stage 1's build_backbone() -- NOTE GatedFusionLight is intentionally NOT a case
# here: it is handled via FUSION_MODELS below and hardcoded to img_size=224 in train_one(),
# matching how it was actually trained.
def build_backbone(name, num_classes, pretrained=USE_PRETRAINED):
    if name == "MobileNetV3Large":
        m = torchvision.models.mobilenet_v3_large(weights="IMAGENET1K_V2" if pretrained else None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
        return m, IMG_SIZE_CNN, 24
    if name == "EfficientNetB0":
        return timm.create_model("efficientnet_b0", pretrained=pretrained, num_classes=num_classes), IMG_SIZE_CNN, 24
    if name == "ResNet50":
        m = torchvision.models.resnet50(weights="IMAGENET1K_V2" if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m, IMG_SIZE_CNN, 16
    if name == "EfficientNetV2S":
        return timm.create_model("tf_efficientnetv2_s", pretrained=pretrained, num_classes=num_classes), IMG_SIZE_CNN, 12
    if name == "ConvNeXtV2Tiny":
        return timm.create_model("convnextv2_tiny", pretrained=pretrained, num_classes=num_classes), IMG_SIZE_CNN, 16
    if name == "ViTBase16":
        return timm.create_model("vit_base_patch16_224", pretrained=pretrained, num_classes=num_classes), 224, 16
    if name == "SwinTiny":
        return timm.create_model("swin_tiny_patch4_window7_224", pretrained=pretrained, num_classes=num_classes), 224, 16
    raise ValueError(name)

class _Gated(nn.Module):
    def __init__(self, ba, da, bb, db, nc, gh, hh, dr):
        super().__init__(); self.branch_a=ba; self.branch_b=bb; fused=da+db
        self.gate = nn.Sequential(nn.Linear(fused,gh), nn.ReLU(), nn.Linear(gh,2), nn.Softmax(dim=1))
        self.classifier = nn.Sequential(nn.Linear(fused,hh), nn.ReLU(), nn.Dropout(dr), nn.Linear(hh,nc))
    def forward(self, x):
        fa, fb = self.branch_a(x), self.branch_b(x)
        g = self.gate(torch.cat([fa,fb], 1))
        return self.classifier(torch.cat([fa*g[:,0:1], fb*g[:,1:2]], 1))

def GatedFusionLightModel(nc, pretrained=USE_PRETRAINED):
    mb = torchvision.models.mobilenet_v3_large(weights="IMAGENET1K_V2" if pretrained else None)
    da = mb.classifier[0].in_features; mb.classifier = nn.Identity()
    b  = timm.create_model("efficientnet_b0", pretrained=pretrained, num_classes=0, global_pool="avg")
    return _Gated(mb, da, b, b.num_features, nc, 128, 256, 0.3)

FUSION_MODELS = {"GatedFusionLight": GatedFusionLightModel}

In [6]:
# ============ CLASSIFIER SECTION: TRAIN / METRICS HELPERS ============
def train_one(model_name, num_epochs=NUM_EPOCHS, patience=PATIENCE):
    if model_name in FUSION_MODELS:
        model = FUSION_MODELS[model_name](NUM_CLASSES).to(device); img_size, bs = 224, 16
    else:
        model, img_size, bs = build_backbone(model_name, NUM_CLASSES); model = model.to(device)
    ttf, etf = make_transforms(img_size)
    tl = DataLoader(DS(train_df, ttf), batch_size=bs, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
    vl = DataLoader(DS(val_df,   etf), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    te = DataLoader(DS(test_df,  etf), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    is_tf = model_name in ("ViTBase16","SwinTiny")
    lr = 5e-5 if is_tf else LR; warm_ep = 3 if is_tf else 2; pat = 10 if is_tf else patience
    opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    warm = torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=warm_ep)
    cos  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, num_epochs-warm_ep))
    sch  = torch.optim.lr_scheduler.SequentialLR(opt, [warm, cos], milestones=[warm_ep])
    crit = nn.CrossEntropyLoss(weight=class_weights if USE_CLASS_WEIGHTS else None, label_smoothing=LABEL_SMOOTHING)
    scaler = torch.cuda.amp.GradScaler(enabled=device.type=="cuda")
    hist = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}; best=-1; best_state=None; bad=0
    for ep in range(num_epochs):
        model.train(); run=cor=tot=0
        for x,y in tl:
            x,y = x.to(device),y.to(device); opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=device.type=="cuda"):
                out = model(x); loss = crit(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item()*x.size(0); cor += (out.argmax(1)==y).sum().item(); tot += y.size(0)
        trl, tra = run/tot, cor/tot
        model.eval(); vr=vc=vt=0
        with torch.no_grad():
            for x,y in vl:
                x,y = x.to(device),y.to(device); out = model(x); loss = crit(out,y)
                vr += loss.item()*x.size(0); vc += (out.argmax(1)==y).sum().item(); vt += y.size(0)
        vlo, vac = vr/vt, vc/vt; sch.step()
        for k,v in zip(hist, [trl,vlo,tra,vac]): hist[k].append(v)
        print(f"[{model_name}] ep{ep+1}/{num_epochs} lr={opt.param_groups[0]['lr']:.1e} "
              f"trL={trl:.3f} vL={vlo:.3f} trA={tra:.3f} vA={vac:.3f}")
        if vac > best+1e-4:
            best=vac; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad += 1
            if ep >= warm_ep and bad >= pat:
                print(f"  early stop @ ep{ep+1}"); break
    model.load_state_dict(best_state); torch.save(best_state, OUTDIR/f"{model_name}_seed{SEED}_best.pt")
    model.eval(); P=[]; Y=[]; PR=[]
    with torch.no_grad():
        for x,y in te:
            x = x.to(device); lg = model(x)
            if USE_TTA: lg = lg + model(torch.flip(x, dims=[3]))
            p = F.softmax(lg, 1).cpu().numpy(); P.extend(p.argmax(1)); Y.extend(y.numpy()); PR.extend(p)
    return model, img_size, hist, np.array(Y), np.array(P), np.array(PR)

def deployment(model, img_size, nw=5, ni=30):
    n = sum(p.numel() for p in model.parameters())
    tmp = OUTDIR/"_t.pt"; torch.save(model.state_dict(), tmp)
    sz = os.path.getsize(tmp)/1e6; os.remove(tmp)
    d  = torch.randn(1, 3, img_size, img_size)
    def t(dev):
        m = model.to(dev).eval(); dd = d.to(dev)
        with torch.no_grad():
            for _ in range(nw): m(dd)
            if dev.type=="cuda": torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(ni): m(dd)
            if dev.type=="cuda": torch.cuda.synchronize()
            return (time.time()-t0)/ni*1000
    lg = t(device) if device.type=="cuda" else float("nan")
    lc = t(torch.device("cpu")); model.to(device)
    return {"params_M": n/1e6, "checkpoint_size_MB": sz, "latency_cpu_ms": lc, "latency_gpu_ms": lg}

def compute_metrics(y, yp, ypb, n):
    cm = confusion_matrix(y, yp, labels=range(n)); specs=[]
    for i in range(n):
        tn = cm.sum()-cm[i,:].sum()-cm[:,i].sum()+cm[i,i]; fp = cm[:,i].sum()-cm[i,i]
        specs.append(tn/(tn+fp) if (tn+fp)>0 else float("nan"))
    present = [c for c in range(n) if (y==c).sum()>0]; aucs=[]
    for c in present:
        try: aucs.append(roc_auc_score((y==c).astype(int), ypb[:,c]))
        except Exception: pass
    return {"accuracy": accuracy_score(y,yp),
            "balanced_accuracy": balanced_accuracy_score(y,yp),
            "f1_macro": f1_score(y,yp,average="macro"),
            "f1_weighted": f1_score(y,yp,average="weighted"),
            "precision_macro": precision_score(y,yp,average="macro",zero_division=0),
            "recall_macro": recall_score(y,yp,average="macro",zero_division=0),
            "mcc": matthews_corrcoef(y,yp),
            "roc_auc_macro": float(np.mean(aucs)) if aucs else float("nan")}, cm

In [7]:
# ============ CLASSIFIER SECTION: MAIN LOOP (all 8 classifiers, SEED 123) ============
for MODEL_NAME in MODELS_TO_TRAIN:
    print(f"\n{'='*60}\n  Training: {MODEL_NAME}  (seed {SEED})\n{'='*60}\n")
    t0 = time.time()
    model, img_size, hist, Y, P, PR = train_one(MODEL_NAME)
    train_time = time.time()-t0
    metrics, cm = compute_metrics(Y, P, PR, NUM_CLASSES)
    dep = deployment(model, img_size)
    result = {"model_name": MODEL_NAME, "seed": SEED, "history": hist,
              "y_true": Y, "y_pred": P, "y_prob": PR,
              "metrics": metrics, "cm": cm, "deployment": dep,
              "img_size": img_size, "train_time_sec": train_time, "class_names": CLASS_NAMES}
    pickle.dump(result, open(OUTDIR/f"result_{MODEL_NAME}_seed{SEED}.pkl", "wb"))
    print("SAVED", OUTDIR/f"result_{MODEL_NAME}_seed{SEED}.pkl")
    print({k: round(v,4) for k,v in metrics.items() if isinstance(v, float)})
    print(f"  train_time: {train_time/3600:.2f}h")
print("\nAll 8 classifiers done for this lane (seed 123).")


  Training: ConvNeXtV2Tiny  (seed 123)



model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

[ConvNeXtV2Tiny] ep1/40 lr=1.6e-04 trL=4.186 vL=3.261 trA=0.262 vA=0.485
[ConvNeXtV2Tiny] ep2/40 lr=3.0e-04 trL=2.595 vL=2.354 trA=0.511 vA=0.625
[ConvNeXtV2Tiny] ep3/40 lr=3.0e-04 trL=2.393 vL=2.450 trA=0.529 vA=0.566
[ConvNeXtV2Tiny] ep4/40 lr=3.0e-04 trL=1.918 vL=2.382 trA=0.642 vA=0.597
[ConvNeXtV2Tiny] ep5/40 lr=3.0e-04 trL=1.634 vL=2.128 trA=0.714 vA=0.688
[ConvNeXtV2Tiny] ep6/40 lr=2.9e-04 trL=1.472 vL=2.074 trA=0.772 vA=0.695
[ConvNeXtV2Tiny] ep7/40 lr=2.9e-04 trL=1.380 vL=2.142 trA=0.794 vA=0.689
[ConvNeXtV2Tiny] ep8/40 lr=2.8e-04 trL=1.272 vL=2.096 trA=0.828 vA=0.683
[ConvNeXtV2Tiny] ep9/40 lr=2.8e-04 trL=1.255 vL=2.098 trA=0.846 vA=0.716
[ConvNeXtV2Tiny] ep10/40 lr=2.7e-04 trL=1.166 vL=2.160 trA=0.876 vA=0.683
[ConvNeXtV2Tiny] ep11/40 lr=2.6e-04 trL=1.159 vL=2.286 trA=0.874 vA=0.662
[ConvNeXtV2Tiny] ep12/40 lr=2.5e-04 trL=1.160 vL=2.116 trA=0.868 vA=0.694
[ConvNeXtV2Tiny] ep13/40 lr=2.4e-04 trL=1.139 vL=2.206 trA=0.888 vA=0.678
[ConvNeXtV2Tiny] ep14/40 lr=2.3e-04 trL=1.128 v

model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

[EfficientNetV2S] ep1/40 lr=1.6e-04 trL=5.149 vL=4.615 trA=0.040 vA=0.117
[EfficientNetV2S] ep2/40 lr=3.0e-04 trL=3.320 vL=2.638 trA=0.359 vA=0.552
[EfficientNetV2S] ep3/40 lr=3.0e-04 trL=2.372 vL=2.545 trA=0.553 vA=0.598
[EfficientNetV2S] ep4/40 lr=3.0e-04 trL=1.942 vL=2.415 trA=0.658 vA=0.641
[EfficientNetV2S] ep5/40 lr=3.0e-04 trL=1.687 vL=2.339 trA=0.733 vA=0.651
[EfficientNetV2S] ep6/40 lr=2.9e-04 trL=1.522 vL=2.293 trA=0.789 vA=0.684
[EfficientNetV2S] ep7/40 lr=2.9e-04 trL=1.455 vL=2.209 trA=0.807 vA=0.681
[EfficientNetV2S] ep8/40 lr=2.8e-04 trL=1.362 vL=2.197 trA=0.842 vA=0.699
[EfficientNetV2S] ep9/40 lr=2.8e-04 trL=1.310 vL=2.222 trA=0.856 vA=0.701
[EfficientNetV2S] ep10/40 lr=2.7e-04 trL=1.296 vL=2.267 trA=0.870 vA=0.694
[EfficientNetV2S] ep11/40 lr=2.6e-04 trL=1.253 vL=2.206 trA=0.883 vA=0.708
[EfficientNetV2S] ep12/40 lr=2.5e-04 trL=1.239 vL=2.240 trA=0.890 vA=0.693
[EfficientNetV2S] ep13/40 lr=2.4e-04 trL=1.176 vL=2.258 trA=0.911 vA=0.689
[EfficientNetV2S] ep14/40 lr=2.3e-

...

[MobileNetV3Large] ep1/40 lr=1.6e-04 trL=4.939 vL=4.882 trA=0.048 vA=0.151
[MobileNetV3Large] ep2/40 lr=3.0e-04 trL=3.532 vL=2.817 trA=0.306 vA=0.483
[MobileNetV3Large] ep3/40 lr=3.0e-04 trL=2.537 vL=2.822 trA=0.497 vA=0.472
[MobileNetV3Large] ep4/40 lr=3.0e-04 trL=2.017 vL=2.383 trA=0.614 vA=0.589
[MobileNetV3Large] ep5/40 lr=3.0e-04 trL=1.706 vL=2.457 trA=0.701 vA=0.613
[MobileNetV3Large] ep6/40 lr=2.9e-04 trL=1.579 vL=2.298 trA=0.746 vA=0.632
[MobileNetV3Large] ep7/40 lr=2.9e-04 trL=1.465 vL=2.343 trA=0.786 vA=0.632
[MobileNetV3Large] ep8/40 lr=2.8e-04 trL=1.374 vL=2.263 trA=0.813 vA=0.656
[MobileNetV3Large] ep9/40 lr=2.8e-04 trL=1.290 vL=2.191 trA=0.840 vA=0.681
[MobileNetV3Large] ep10/40 lr=2.7e-04 trL=1.269 vL=2.219 trA=0.859 vA=0.684
[MobileNetV3Large] ep11/40 lr=2.6e-04 trL=1.228 vL=2.167 trA=0.870 vA=0.681
[MobileNetV3Large] ep12/40 lr=2.5e-04 trL=1.158 vL=2.145 trA=0.895 vA=0.701
[MobileNetV3Large] ep13/40 lr=2.4e-04 trL=1.153 vL=2.202 trA=0.896 vA=0.686
[MobileNetV3Large] ep

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

[ViTBase16] ep1/40 lr=2.0e-05 trL=4.775 vL=4.102 trA=0.080 vA=0.253
[ViTBase16] ep2/40 lr=3.5e-05 trL=2.764 vL=2.370 trA=0.462 vA=0.625
[ViTBase16] ep3/40 lr=5.0e-05 trL=1.875 vL=2.275 trA=0.669 vA=0.655
[ViTBase16] ep4/40 lr=5.0e-05 trL=1.720 vL=2.483 trA=0.715 vA=0.623
[ViTBase16] ep5/40 lr=5.0e-05 trL=1.408 vL=2.180 trA=0.806 vA=0.678
[ViTBase16] ep6/40 lr=4.9e-05 trL=1.283 vL=2.213 trA=0.845 vA=0.677
[ViTBase16] ep7/40 lr=4.9e-05 trL=1.225 vL=2.290 trA=0.879 vA=0.675
[ViTBase16] ep8/40 lr=4.8e-05 trL=1.193 vL=2.248 trA=0.889 vA=0.664
[ViTBase16] ep9/40 lr=4.7e-05 trL=1.149 vL=2.192 trA=0.907 vA=0.682
[ViTBase16] ep10/40 lr=4.6e-05 trL=1.085 vL=2.151 trA=0.915 vA=0.696
[ViTBase16] ep11/40 lr=4.4e-05 trL=1.051 vL=2.115 trA=0.932 vA=0.719
[ViTBase16] ep12/40 lr=4.3e-05 trL=1.064 vL=2.242 trA=0.925 vA=0.688
[ViTBase16] ep13/40 lr=4.2e-05 trL=1.038 vL=2.179 trA=0.935 vA=0.714
[ViTBase16] ep14/40 lr=4.0e-05 trL=1.047 vL=2.166 trA=0.931 vA=0.701
[ViTBase16] ep15/40 lr=3.8e-05 trL=0.996 vL

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

[SwinTiny] ep1/40 lr=2.0e-05 trL=4.994 vL=5.074 trA=0.017 vA=0.053
[SwinTiny] ep2/40 lr=3.5e-05 trL=4.142 vL=3.374 trA=0.195 vA=0.378
[SwinTiny] ep3/40 lr=5.0e-05 trL=2.810 vL=2.686 trA=0.459 vA=0.539
[SwinTiny] ep4/40 lr=5.0e-05 trL=2.213 vL=2.432 trA=0.588 vA=0.612
[SwinTiny] ep5/40 lr=5.0e-05 trL=1.846 vL=2.355 trA=0.690 vA=0.606
[SwinTiny] ep6/40 lr=4.9e-05 trL=1.625 vL=2.238 trA=0.744 vA=0.664
[SwinTiny] ep7/40 lr=4.9e-05 trL=1.483 vL=2.234 trA=0.789 vA=0.661
[SwinTiny] ep8/40 lr=4.8e-05 trL=1.396 vL=2.248 trA=0.821 vA=0.675
[SwinTiny] ep9/40 lr=4.7e-05 trL=1.338 vL=2.165 trA=0.843 vA=0.687
[SwinTiny] ep10/40 lr=4.6e-05 trL=1.275 vL=2.189 trA=0.864 vA=0.693
[SwinTiny] ep11/40 lr=4.4e-05 trL=1.222 vL=2.188 trA=0.873 vA=0.696
[SwinTiny] ep12/40 lr=4.3e-05 trL=1.190 vL=2.123 trA=0.892 vA=0.707
[SwinTiny] ep13/40 lr=4.2e-05 trL=1.151 vL=2.154 trA=0.898 vA=0.709
[SwinTiny] ep14/40 lr=4.0e-05 trL=1.120 vL=2.102 trA=0.918 vA=0.707
[SwinTiny] ep15/40 lr=3.8e-05 trL=1.112 vL=2.153 trA=0.91

## Classifiers-Only Lane A (seed 123) complete

All 8 classifiers trained and saved to `/kaggle/working/s1_results_seed123/`. Bring the
`result_*_seed123.pkl` files (and `*_best.pt` checkpoints, if you want them) back to
`revision_work/multiseed/` for aggregation with the already-collected seed-123 segmenter
results and the original SEED=42 checkpoints.